In [ ]:
import numpy as np
import pandas as pd
import joblib
from d3rlpy.algos import DiscreteCQLConfig

ACTION_MAP = {
    'воспользовался поиском': 'SEARCH',
    'воспользовался каталогом': 'CATALOG',
    'находится в карточке товара': 'CARD_VIEW',
    'просматривает сертификат товара': 'CARD_VIEW',
    'добавил в корзину': 'CART_VIEW',
    'находится в корзине': 'CART_VIEW',
    'оформление заказа': 'CART_VIEW',
    'подтверждение заказа': 'CART_VIEW',
    'оплата': 'CART_VIEW',
    'находится на странице акций': 'PROMO',
    'нажал применить промокод': 'PROMO',
    'рассрочка': 'PROMO',
    'распродажа': 'PROMO',
    'цены снижены': 'PROMO',
    'находится в списке статей': 'ARTICLES',
    'находится на главной странице': 'MAIN',
    'личный кабинет': 'CABINET',
    'избранное': 'CABINET',
}

CLASS_LIST = sorted(set(ACTION_MAP.values()))
CART_ID = CLASS_LIST.index("CART_VIEW")

def map_action(text):
    t = str(text).lower()
    for key, cls in ACTION_MAP.items():
        if key in t:
            return cls
    return 'OTHER'

def build_features(df):
    df = df.copy()
    df.sort_values(['sess_id', 'watch_tms'], inplace=True)
    df['action'] = df['goal_nm_lvl1'].apply(map_action)
    df['watch_tms'] = pd.to_datetime(df['watch_tms'])
    df['delta_sec'] = df.groupby('sess_id')['watch_tms'].diff().dt.total_seconds().fillna(0)
    df['delta_sec'] = df['delta_sec'].clip(0, 3600)
    df['step_idx'] = df.groupby('sess_id').cumcount()
    df['sess_len'] = df.groupby('sess_id')['action'].transform('size')
    df['unique_actions'] = df.groupby('sess_id')['action'].transform('nunique')
    df['sess_time'] = df.groupby('sess_id')['delta_sec'].transform('sum')
    df['time_from_start'] = df.groupby('sess_id')['delta_sec'].cumsum()
    df['time_to_end'] = df['sess_time'] - df['time_from_start']
    df['act_id'] = pd.factorize(df['action'])[0]
    feat_cols = [
        'act_id', 'delta_sec',
        'step_idx', 'sess_len', 'unique_actions',
        'sess_time', 'time_from_start', 'time_to_end'
    ]
    return df, feat_cols

hmm_pos = joblib.load("../models/hmm_pos.pkl")
hmm_neg = joblib.load("../models/hmm_neg.pkl")

cfg = DiscreteCQLConfig(batch_size=128)
cql = cfg.create(device="cpu")
cql.build_with_obs_shape((1,))
cql.load_model("../models/cql_model.d3")

def session_score(seq):
    ll_pos = hmm_pos.score(seq)
    ll_neg = hmm_neg.score(seq)
    return 1 / (1 + np.exp(-(ll_pos - ll_neg)))

def recommend(df, sess_id, threshold=0.25):
    df_feat, feat_cols = build_features(df)
    seq = df_feat[df_feat['sess_id'] == sess_id][feat_cols].values
    p = session_score(seq)
    if p >= threshold:
        return {"p_cart": p, "recommendation": None}
    state = hmm_pos.predict(seq)[-1]
    actions = np.arange(len(CLASS_LIST))
    obs = np.full((len(actions), 1), state)
    q_vals = cql.predict_value(obs, actions)
    best = q_vals.argmax()
    if best == CART_ID:
        best = np.argsort(q_vals)[-2]
    return {"p_cart": p, "recommendation": CLASS_LIST[best]}

df = pd.read_csv("../data/actions.csv", sep=";")
sess = df['sess_id'].iloc[0]
print(recommend(df, sess))